# វិភាគការស្នើរសុំចំណាយ

កំណត់ត្រានេះបង្ហាញពីវិធីបង្កើតភ្នាក់ងារដែលប្រើប្រាស់ផ្លាកអជើញឧបករណ៍ដើម្បីដំណើរការចំណាយការធ្វើដំណើរពីរូបភាពបង្កាន់ដៃក្នុងតំបន់, បង្កើតអ៊ីមែលស្នើរសុំចំណាយ, និងបង្ហាញទិន្នន័យចំណាយដោយប្រើតាប្លូប៊ីស។ ភ្នាក់ងារជ្រើសរើសមុខងារដោយឥតប្រាក់បង់ដោយផ្អែកលើបរិបទភារកិច្ច។

ជំហាន៖
1. ភ្នាក់ងារ OCR ដំណើរការរូបភាពបង្កាន់ដៃក្នុងតំបន់ ហើយបង្កើតទិន្នន័យចំណាយការធ្វើដំណើរ។
2. ភ្នាក់ងារ អ៊ីមែល បង្កើតអ៊ីមែលស្នើរសុំចំណាយ។

### ឧទាហរណ៍នៃសេណារីយ៉ូចំណាយការធ្វើដំណើរ៖
សូមនឹកស្រមៃថាអ្នកជាបុគ្គលិកម្នាក់ធ្វើដំណើរសម្រាប់កិច្ចប្រជុំអាជីវកម្មនៅទីក្រុងផ្សេងទៀត។ ក្រុមហ៊ុនរបស់អ្នកមានគោលការណ៍ដែលត្រូវសងប្រាក់ត្រឡប់វិញចំពោះចំណាយទាក់ទងនឹងការធ្វើដំណើរដែលសមរម្យទាំងអស់។ នេះគឺជាការបំបែកចំណាយការធ្វើដំណើរដែលអាចមាន៖
- ការដឹកជញ្ជូន៖
សំបុត្រយន្តហោះប្រវត្តិជំរើសចេញពីទីក្រុងរស់នៅរបស់អ្នកទៅទីក្រុងគោលដៅ។
រថយន្តលីបឡូត ការសេវាប៉ាក់ស៊ី ឬសេវាដឹកជញ្ជូនទៅមកអាកាសយានដ្ឋាន។
ការដឹកជញ្ជូនក្នុងតំបន់ទីក្រុងគោលដៅ (ដូចជារថយន្តសាធារណៈ, រថយន្តជួល, ឬប៉ាក់ស៊ី)។

- ការស្នាក់នៅ៖
ការស្នាក់នៅសណ្ឋាគារសម្រាប់បីយប់នៅសណ្ឋាគារអាជីវកម្មមធ្យមកំពស់ជិតកន្លែងប្រជុំ។

- អាហារ៖
ចំនួនប្រាក់អាហារប្រចាំថ្ងៃសម្រាប់អាហារព្រឹក អាហារថ្ងៃត្រង់ និងអាហារថ្ងៃល្ងាច, ដោយផ្អែកលើគោលការណ៍ per diem របស់ក្រុមហ៊ុន។

- ចំណាយផ្សេងៗ៖
កម្រៃចតយានយន្តនៅអាកាសយានដ្ឋាន។
ការបង់ប្រាក់ស្វាគមន៍អ៊ីនធឺណិតនៅសណ្ឋាគារ។
រង្វាន់ ឬចំណាយសេវាថ្នាក់តូចៗ។

- ការបញ្ចូនឯកសារ៖
អ្នកដាក់ស្នើបង្កាន់ដៃទាំងអស់ (សំបុត្រយន្តហោះ, ប៉ាក់ស៊ី, សណ្ឋាគារ, អាហារ, ល។) និងរបាយការណ៍ចំណាយដែលបំពេញរួចសម្រាប់សំណងប្រាក់វិញ។


## នាំចូលបណ្ណាល័យដែលត្រូវការ

នាំចូលបណ្ណាល័យ និងម៉ូឌុលដែលចាំបាច់សម្រាប់សៀវភៅកំណត់ត្រា។


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Azure AI Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## កំណត់ម៉ូដែលចំណាយ

 បង្កើតម៉ូដែល Pydantic សម្រាប់ចំណាយឯកតារៀងៗខ្លួន និងថ្នាក់ ExpenseFormatter ដើម្បីបម្លែងសំណួររបស់អ្នកប្រើទៅជាទិន្នន័យចំណាយដែលមានរចនាសម្ព័ន្ធ។

 ចំណាយនិងតំណាងដោយទ្រង់ទ្រាយ៖  
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## ការកំណត់ឧបករណ៍ - ការបង្កើតអ៊ីមែល

បង្កើតមុខងារ​ឧបករណ៍​មួយ​សម្រាប់​បង្កើត​អ៊ីមែល​សម្រាប់​បញ្ជូន​ការ​ស្នើសុំ​ចំណាយ។  
- ឧបករណ៍នេះ​ប្រើ `@tool` decorator ពី Microsoft Agent Framework។  
- វាគណនាចំនួនទឹកប្រាក់សរុបនៃការចំណាយ និង​បង្កើតរាងរបរ​លម្អិត​ទៅជា​ខ្លឹមសារ​អ៊ីមែល។


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# ឧបករណ៍សម្រាប់ដកចំនាយដំណើរកំសាន្តពីរូបភាពវិក្កយបត្រ

បង្កើតមុខងារឧបករណ៍ដើម្បីដកចំណាយដំណើរកំសាន្តពីរូបភាពវិក្កយបត្រ។
- ឧបករណ៍នេះប្រើ `@tool` decorator ពី Microsoft Agent Framework។
- វាអានរូបភាពវិក្កយបត្រ, បម្លែងវាជា base64, ហើយបញ្ជូន URI ទិន្នន័យសម្រាប់ភ្នាក់ងារវិភាគ។


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## ការដំណើរការចំណាយ

កំណត់ភ្នាក់ងារ ហើយភ្ជាប់ពួកវាទៅក្នុងលំហូរឧបករណ៍តាមលំដាប់ដោយប្រើ `WorkflowBuilder`។
- ភ្នាក់ងារ OCR ទាញយកទិន្នន័យចំណាយដែលមានរចនាសម្ព័ន្ធពីរូបភាពបង្កាន់ដៃដោយប្រើឧបករណ៍ `load_receipt_image`។
- ភ្នាក់ងារ Email ទទួលយកទិន្នន័យដែលបានទាញយក ហើយបង្កើតអ៊ីមែលអត្ថប្រយោជន៍ចំណាយដែលមានវិជ្ជាជីវៈដោយប្រើឧបករណ៍ `generate_expense_email`។
- `WorkflowBuilder` ជាមួយ `add_edge` បង្កើតបំពង់រង្វិលតាមលំដាប់: ភ្នាក់ងារ OCR → ភ្នាក់ងារ Email។


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## មុខងារសំខាន់

សាងសង់ដំណើរការតាមលំដាប់ជាប់គ្នា ហើយរត់វាដើម្បីដំណើរការរូបភាពបង្កាន់ដៃ និងបង្កើតអ៊ីមែលអះអាងចំណាយ។


> **កំណត់ចំណាំៈ** ប្រតិបត្តិការនេះបច្ចុប្បន្នផ្ញើរូបភាពវិក័យប័ត្រ ក្នុងទ្រង់ទ្រាយអក្សរបែប base64 ដែលគំរូជជែកភ្លោះច្រើន (រួមមាន gpt-4o) មិនបានរក្សាទុកជារូបភាពទេ។  
> វាអាចលើសព្រំប្រទល់បរិបទគំរូបានផងដែរ។ ល្អប្រសើរជាងគេក្នុងការរត់ OCR ជាមួយ Azure AI Vision (ឬឧបករណ៍ OCR ផ្សេងទៀត) ហើយផ្ញើតែអត្ថបទដែលបានដកចេញ ឬកែប្រែដើម្បីផ្ញើរូបភាពជាសារប្រើបញ្ចូល `image_url`។  
> ប្រសិនបើអ្នកចង់ជៀសវាងកំហុសបរិបទសូមសាកល្បងរូបភាពវិក័យប័ត្រតិចជាងឬគំរូដែលមានព្រំប្រទល់បរិបទធំជាង។


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ការបដិសេធ**:
ឯកសារនេះត្រូវបានបម្លែងភាសា ដោយប្រើសេវាបម្លែងភាសា AI [Co-op Translator](https://github.com/Azure/co-op-translator)។ ទោះយើងខ្ញុំមានក្តីប្រាថ្នាឱ្យបានច្បាស់លាស់ តែសូមយល់ដឹងថាការបម្លែងដោយស្វ័យប្រវត្តិក៏អាចមានកំហុសឬភាពមិនត្រឹមត្រូវ។ ឯកសារដើមជាភាសាទីតាំងគួរត្រូវបានគេប្រើជាប្រភពច្បាស់លាស់។ សម្រាប់ព័ត៌មានសំខាន់ៗ សូមណែនាំឱ្យប្រើប្រាស់ការប្រែដោយមនុស្សជំនាញ។ យើងខ្ញុំមិនទទួលខុសត្រូវចំពោះការយល់ច្រឡំ ឬការបកស្រាយខុសបន្ទាប់ពីការប្រើប្រាស់ការបម្លែងនេះនោះទេ។
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
